In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import holoviews as hv
import hvplot.pandas
import panel as pn
from tqdm import tqdm
from concurrent.futures import ProcessPoolExecutor, as_completed
import multiprocessing as mp

from scipy import stats
from pathlib import Path
from pprint import pprint
from holoviews import opts
from bokeh.io import output_notebook


output_notebook()
hv.extension('bokeh')

font_dict = {'title': 16, 'labels': 14, 'ticks': 12, 'legend': 12}
hv.opts.defaults(
    hv.opts.Curve(width=600, height=400, tools=['hover'], fontsize=font_dict),
    hv.opts.Scatter(width=600, height=400, size=8, tools=['hover'], fontsize=font_dict),
    hv.opts.Histogram(width=600, height=400, fontsize=font_dict),
    hv.opts.Bars(width=600, height=400, fontsize=font_dict),
)

Loading BokehJS ...

In [2]:
monkey = 'fiona' # 'yasmin'  or 'fiona' 
base_path = Path.cwd().parent / 'data' / 'csst_trials_pkls'
# filepath = base_path / f'all_{monkey}_CSST_trials_df.pkl'
filepath = base_path / f'all_{monkey}_CSST_trials_df_including_all_200_neuron_fields.pkl'

df = pd.read_pickle(filepath)

print(df.info())
# df.iloc[:2]
df.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 110358 entries, 0 to 110357
Data columns (total 28 columns):
 #   Column                  Non-Null Count   Dtype  
---  ------                  --------------   -----  
 0   blinks                  21339 non-null   object 
 1   dir                     110358 non-null  int64  
 2   direction               110358 non-null  object 
 3   filename                110358 non-null  object 
 4   first_relevant_saccade  103093 non-null  object 
 5   flags                   110358 non-null  int64  
 6   go_cue                  110358 non-null  int64  
 7   hPos                    110358 non-null  object 
 8   hVel                    110358 non-null  object 
 9   neural_data             110358 non-null  object 
 10  reaction_time           103093 non-null  float64
 11  saccades                110206 non-null  object 
 12  screen_rotation         110358 non-null  float64
 13  segs_durations          110358 non-null  object 
 14  segs_times          

,blinks,dir,direction,filename,first_relevant_saccade,flags,go_cue,hPos,hVel,neural_data,...,ssd_number,stop_cue,trial_failed,trial_length,trial_name,trial_number,trial_session,type,vPos,vVel
0,None,180,L,fi210824a.0614,"[1382, 1457]",8206,1054,"[11.275, 11.275, 11.275, 11.275, 11.275, 11.27...","[0.0, 0.0, 0.4594490287247533, 1.3783470861742...","{0: [884.4], 1: [154.42, 329.18, 1478.9], 2: [...",...,2.0,1186.0,False,2205,CONT_L_SSD2,0614,fi210824a,CONT,"[-0.05, -0.05, -0.05, -0.05, -0.05, -0.05, 0.0...","[-3.1242533953283225, -3.1242533953283225, -4...."
1,None,180,L,fi210824a.0520,"[1100, 1172]",8206,914,"[-11.15, -11.15, -11.15, -11.15, -11.15, -11.1...","[-2.7566941723485194, -2.7566941723485194, -3....","{0: [], 1: [48.52, 264.55, 585.97, 1032.3], 2:...",...,NaN,NaN,False,2065,GO_L,0520,fi210824a,GO,"[-1.15, -1.15, -1.15, -1.15, -1.15, -1.175, -1...","[-0.8270082517045559, -0.8270082517045559, 0.1..."
2,None,180,L,fi210824a.1193,"[1099, 1179]",8206,938,"[2.25, 2.25, 2.3, 2.3, 2.3, 2.25, 2.225, 2.225...","[-188.19032216565896, -188.19032216565896, -18...","{0: [], 1: [574.7, 853.02, 1403.57], 2: [], 3:...",...,3.0,1118.0,False,2089,CONT_L_SSD3,1193,fi210824a,CONT,"[-27.45, -27.45, -27.45, -27.45, -27.45, -27.0...","[0.0, 0.0, 0.0, 0.0, 0.0, 9.280870380240016, 4..."
3,None,180,L,fi210824a.1013,"[1213, 1289]",13326,1081,"[8.425, 8.425, 8.375, 8.375, 8.375, 8.4, 8.4, ...","[1.286457280429309, 1.286457280429309, -0.9188...","{0: [], 1: [], 2: [1954.73], 3: [], 4: [], 5: ...",...,3.0,1261.0,True,1961,STOP_L_SSD3,1013,fi210824a,STOP,"[0.275, 0.275, 0.275, 0.275, 0.275, 0.275, 0.2...","[0.9188980574495066, 0.9188980574495066, 0.0, ..."
4,None,0,R,fi210824a.1257,NaN,8194,1024,"[-11.475, -11.475, -11.475, -11.475, -11.475, ...","[3.767482035542977, 3.767482035542977, 3.85937...","{0: [923.35], 1: [1125], 2: [], 3: [], 4: [], ...",...,2.0,1156.0,True,1476,CONT_R_SSD2,1257,fi210824a,CONT,"[-1.825, -1.825, -1.8, -1.8, -1.825, -1.825, -...","[0.27566941723485194, 0.27566941723485194, 1.0..."


In [3]:
df['filename'].apply(lambda x: x.split('.')[0][-1]).unique()

array(['a'], dtype=object)

In [4]:
neurons_list = []

def extract_neuron_ids(row, neurons_list):
    trial_neuron_ids = [f"{row['trial_session']}_{key}" for key in row['neural_data'].keys()]
    neurons_list += trial_neuron_ids
    return trial_neuron_ids

df.apply(
    lambda row: extract_neuron_ids(row, neurons_list), 
    axis=1
)

neurons_set = set(neurons_list)
print(f'Total unique neurons across all sessions: {len(neurons_set)}') # 17600 - 2889

Total unique neurons across all sessions: 17600


In [5]:
# Drop unnecessary columns
cols_to_drop = [
    'vPos', 'hPos', 'vVel', 'hVel', 'speed',
    'set', 'direction'
]
df.drop(columns=cols_to_drop, inplace=True)
print(f"DataFrame shape after dropping columns: {df.shape}")
df.head()

DataFrame shape after dropping columns: (110358, 21)


,blinks,dir,filename,first_relevant_saccade,flags,go_cue,neural_data,reaction_time,saccades,screen_rotation,...,segs_times,ssd_len,ssd_number,stop_cue,trial_failed,trial_length,trial_name,trial_number,trial_session,type
0,None,180,fi210824a.0614,"[1382, 1457]",8206,1054,"{0: [884.4], 1: [154.42, 329.18, 1478.9], 2: [...",328.0,"[[51, 129], [1382, 1457], [1428, 1465]]",0.0,...,"[0, 500, 1054, 1186, 1504, 2204]",132,2.0,1186.0,False,2205,CONT_L_SSD2,0614,fi210824a,CONT
1,None,180,fi210824a.0520,"[1100, 1172]",8206,914,"{0: [], 1: [48.52, 264.55, 585.97, 1032.3], 2:...",186.0,"[[139, 217], [608, 668], [1100, 1172], [1144, ...",0.0,...,"[0, 500, 914, 1364, 2064]",450,NaN,NaN,False,2065,GO_L,0520,fi210824a,GO
2,None,180,fi210824a.1193,"[1099, 1179]",8206,938,"{0: [], 1: [574.7, 853.02, 1403.57], 2: [], 3:...",161.0,"[[0, 89], [57, 121], [1099, 1179]]",0.0,...,"[0, 500, 938, 1118, 1388, 2088]",180,3.0,1118.0,False,2089,CONT_L_SSD3,1193,fi210824a,CONT
3,None,180,fi210824a.1013,"[1213, 1289]",13326,1081,"{0: [], 1: [], 2: [1954.73], 3: [], 4: [], 5: ...",132.0,"[[232, 308], [945, 1006], [1213, 1289], [1259,...",0.0,...,"[0, 500, 1081, 1261, 1961]",180,3.0,1261.0,True,1961,STOP_L_SSD3,1013,fi210824a,STOP
4,None,0,fi210824a.1257,NaN,8194,1024,"{0: [923.35], 1: [1125], 2: [], 3: [], 4: [], ...",NaN,"[[241, 325], [893, 951]]",0.0,...,"[0, 500, 1024, 1156, 1474, 2174]",132,2.0,1156.0,True,1476,CONT_R_SSD2,1257,fi210824a,CONT


In [6]:
# reorder columns
new_order = [
    'filename', 'trial_name', 'reaction_time', 
    'go_cue', 'stop_cue', 'trial_failed', 
    'first_relevant_saccade', 'segs_durations', 'segs_times',
    'trial_length', 'ssd_len', 'ssd_number',
    'screen_rotation', 'neural_data', 'saccades', 
    'blinks', 'dir', 'flags',
    'type', 'trial_session', 'trial_number',
]

df = df[new_order]
df.head()

,filename,trial_name,reaction_time,go_cue,stop_cue,trial_failed,first_relevant_saccade,segs_durations,segs_times,trial_length,...,ssd_number,screen_rotation,neural_data,saccades,blinks,dir,flags,type,trial_session,trial_number
0,fi210824a.0614,CONT_L_SSD2,328.0,1054,1186.0,False,"[1382, 1457]","[500, 554, 132, 318, 700]","[0, 500, 1054, 1186, 1504, 2204]",2205,...,2.0,0.0,"{0: [884.4], 1: [154.42, 329.18, 1478.9], 2: [...","[[51, 129], [1382, 1457], [1428, 1465]]",None,180,8206,CONT,fi210824a,0614
1,fi210824a.0520,GO_L,186.0,914,NaN,False,"[1100, 1172]","[500, 414, 450, 700]","[0, 500, 914, 1364, 2064]",2065,...,NaN,0.0,"{0: [], 1: [48.52, 264.55, 585.97, 1032.3], 2:...","[[139, 217], [608, 668], [1100, 1172], [1144, ...",None,180,8206,GO,fi210824a,0520
2,fi210824a.1193,CONT_L_SSD3,161.0,938,1118.0,False,"[1099, 1179]","[500, 438, 180, 270, 700]","[0, 500, 938, 1118, 1388, 2088]",2089,...,3.0,0.0,"{0: [], 1: [574.7, 853.02, 1403.57], 2: [], 3:...","[[0, 89], [57, 121], [1099, 1179]]",None,180,8206,CONT,fi210824a,1193
3,fi210824a.1013,STOP_L_SSD3,132.0,1081,1261.0,True,"[1213, 1289]","[500, 581, 180, 700]","[0, 500, 1081, 1261, 1961]",1961,...,3.0,0.0,"{0: [], 1: [], 2: [1954.73], 3: [], 4: [], 5: ...","[[232, 308], [945, 1006], [1213, 1289], [1259,...",None,180,13326,STOP,fi210824a,1013
4,fi210824a.1257,CONT_R_SSD2,NaN,1024,1156.0,True,NaN,"[500, 524, 132, 318, 700]","[0, 500, 1024, 1156, 1474, 2174]",1476,...,2.0,0.0,"{0: [923.35], 1: [1125], 2: [], 3: [], 4: [], ...","[[241, 325], [893, 951]]",None,0,8194,CONT,fi210824a,1257


In [7]:
# load monkey's cell db from xlsx file
cell_db_path = Path.cwd().parent / 'data' / f'database_sst'
cell_db = pd.read_excel(cell_db_path / f'SST_{monkey}_cells_db.xlsx')
print(cell_db.info())
cell_db.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5309 entries, 0 to 5308
Data columns (total 24 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   cell_ID              5309 non-null   int64  
 1   session              5309 non-null   object 
 2   cell_type            5309 non-null   object 
 3   electrode            5309 non-null   int64  
 4   template             5309 non-null   int64  
 5   maestro_ID           5309 non-null   int64  
 6   phy_id               0 non-null      float64
 7   phy_channel          0 non-null      float64
 8   file_begin           5309 non-null   int64  
 9   file_end             5309 non-null   int64  
 10  fb_after_stablility  5309 non-null   object 
 11  fe_after_stability   5309 non-null   object 
 12  plexon_session       5309 non-null   object 
 13  grade                5309 non-null   int64  
 14  X                    5309 non-null   int64  
 15  Y                    5309 non-null   i

,cell_ID,session,cell_type,electrode,template,maestro_ID,phy_id,phy_channel,file_begin,file_end,...,X,Y,depth_mm,is_continuous,comments,plex_sorted_file,tmp,sorted,problem,synced_stability
0,9001,fi210628,ctx,1,1,1,NaN,NaN,1,225,...,0,0,4420.0,2,depth micro m is from cortex surface,fi210628a-01.pl2,NaN,1,broken cell two peaks,1
1,9002,fi210629,msn,1,1,1,NaN,NaN,84,163,...,0,-1,10200.0,2,NaN,fi210629a-01.pl2,NaN,1,NaN,1
2,9003,fi210701,msn,1,1,1,NaN,NaN,16,348,...,0,-1,7030.0,2,2 cells multi unit,fi210701a-02.pl2,NaN,1,NaN,1
3,9004,fi210701,tan,1,1,1,NaN,NaN,350,510,...,0,-1,7110.0,2,NaN,fi210701b-02.pl2,NaN,1,NaN,1
4,9005,fi210701,msn,1,2,2,NaN,NaN,16,348,...,0,-1,7030.0,2,NaN,fi210701a-02.pl2,NaN,1,NaN,1


In [8]:
df.iloc[0].neural_data.keys()

dict_keys([0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89, 90, 91, 92, 93, 94, 95, 96, 97, 98, 99, 100, 101, 102, 103, 104, 105, 106, 107, 108, 109, 110, 111, 112, 113, 114, 115, 116, 117, 118, 119, 120, 121, 122, 123, 124, 125, 126, 127, 128, 129, 130, 131, 132, 133, 134, 135, 136, 137, 138, 139, 140, 141, 142, 143, 144, 145, 146, 147, 148, 149, 150, 151, 152, 153, 154, 155, 156, 157, 158, 159, 160, 161, 162, 163, 164, 165, 166, 167, 168, 169, 170, 171, 172, 173, 174, 175, 176, 177, 178, 179, 180, 181, 182, 183, 184, 185, 186, 187, 188, 189, 190, 191, 192, 193, 194, 195, 196, 197, 198, 199])

In [9]:
cell_db.at[429, 'fe_after_stability'] = 1017

In [10]:
cell_db.columns

Index(['cell_ID', 'session', 'cell_type', 'electrode', 'template',
       'maestro_ID', 'phy_id', 'phy_channel', 'file_begin', 'file_end',
       'fb_after_stablility', 'fe_after_stability', 'plexon_session', 'grade',
       'X', 'Y', 'depth_mm', 'is_continuous', 'comments', 'plex_sorted_file',
       'tmp', 'sorted', 'problem', 'synced_stability'],
      dtype='object')

In [11]:
cell_db[
    cell_db['fe_after_stability'].apply(
        lambda row: isinstance(row, str)
)].apply(
    lambda row: [
        np.fromstring(
            row[key][1:-1], sep=' ', dtype=np.int16
        )
        for key in ['fb_after_stablility', 'fe_after_stability']
    ],
    axis=1, result_type='expand'
)

# np.fromstring(tmp[1:-1], sep=' ', dtype=np.int16)

,0,1
729,"[651, 1163]","[1066, 1279]"
741,"[741, 854]","[814, 949]"
750,"[332, 594]","[519, 644]"
777,"[693, 1235, 1450]","[1123, 1418, 1584]"
795,"[1, 399]","[314, 637]"
...,...,...
3627,"[1, 253]","[116, 630]"
3644,"[1, 254]","[113, 641]"
4149,"[981, 1748]","[1724, 2270]"
4644,"[1054, 1592, 1731, 1965]","[1556, 1700, 1924, 2382]"


In [12]:
def get_row_stable_trials_total(row):
    if isinstance(row['fe_after_stability'], str):
        fb_stable = np.fromstring(
            row['fb_after_stablility'][1:-1], sep=' ', dtype=np.int16
        )
        fe_stable = np.fromstring(
            row['fe_after_stability'][1:-1], sep=' ', dtype=np.int16
        )
        return fe_stable.sum() - fb_stable.sum()
    elif isinstance(row['fe_after_stability'], int):
        return row['fe_after_stability'] - row['fb_after_stablility']
    else:
        raise ValueError("Unexpected data type in 'fe_after_stability' column")
    
row = cell_db.iloc[0]
row = cell_db[
    cell_db['fe_after_stability'].apply(
        lambda row: isinstance(row, str)
)].iloc[0]
get_row_stable_trials_total(row)
# cell_db[(cell_db.apply(get_row_stable_trials_total, axis=1) < 0)]
stable_trials = cell_db[cell_db['cell_type'].isin(['msn', 'pu msn'])].apply(get_row_stable_trials_total, axis=1)
stable_trials.sum()

np.int64(2807528)

In [13]:
# Function to extract session and trial number from filename
def parse_filename(filename):
    """
    Parse filename like 'fi210824a.0614' into components
    Returns: (session, plexon_session, trial_number)
    """
    parts = filename.split('.')
    if len(parts) != 2:
        return None, None, None
    
    prefix = parts[0]  # 'fi210824a'
    trial_num = parts[1]  # '0614'
    
    if len(prefix) < 9:  # minimum: 'fi' + 6 digits + 'a'
        return None, None, None
    
    session = prefix[:-1]  # 'fi210824' (remove plexon session letter)
    plexon_session = prefix[-1]  # 'a'
    
    return session, plexon_session, int(trial_num)

# Test the function
test_filename = df.iloc[0]['filename']
print(f"Test filename: {test_filename}")
session, plexon_session, trial_num = parse_filename(test_filename)
print(f"Parsed: session='{session}', plexon_session='{plexon_session}', trial_num={trial_num}")

# Check a few more examples
print("\nTesting with more filenames:")
for i in range(5):
    fname = df.iloc[i]['filename']
    session, plexon_session, trial_num = parse_filename(fname)
    print(f"{fname} -> session='{session}', plexon_session='{plexon_session}', trial_num={trial_num}")

Test filename: fi210824a.0614
Parsed: session='fi210824', plexon_session='a', trial_num=614

Testing with more filenames:
fi210824a.0614 -> session='fi210824', plexon_session='a', trial_num=614
fi210824a.0520 -> session='fi210824', plexon_session='a', trial_num=520
fi210824a.1193 -> session='fi210824', plexon_session='a', trial_num=1193
fi210824a.1013 -> session='fi210824', plexon_session='a', trial_num=1013
fi210824a.1257 -> session='fi210824', plexon_session='a', trial_num=1257


In [14]:
# Check the column names for stability columns
print("Cell DB columns:")
print(cell_db.columns.tolist())

print(f"\nSample stability columns:")
stability_cols = ['fb_after_stablility', 'fe_after_stability']
for col in stability_cols:
    if col in cell_db.columns:
        print(f"{col}: {cell_db[col].head().tolist()}")
        print(f"  Data types: {cell_db[col].apply(type).value_counts()}")
    else:
        print(f"❌ Column '{col}' not found!")

# Check neural data keys format
print(f"\nSample neural data keys:")
sample_neural_data = df.iloc[0]['neural_data']
print(f"Type: {type(sample_neural_data)}")
if isinstance(sample_neural_data, dict):
    print(f"Keys (first 10): {list(sample_neural_data.keys())[:10]}")
    print(f"Key types: {[type(k) for k in list(sample_neural_data.keys())[:5]]}")
else:
    print(f"Not a dict: {sample_neural_data}")

Cell DB columns:
['cell_ID', 'session', 'cell_type', 'electrode', 'template', 'maestro_ID', 'phy_id', 'phy_channel', 'file_begin', 'file_end', 'fb_after_stablility', 'fe_after_stability', 'plexon_session', 'grade', 'X', 'Y', 'depth_mm', 'is_continuous', 'comments', 'plex_sorted_file', 'tmp', 'sorted', 'problem', 'synced_stability']

Sample stability columns:
fb_after_stablility: [1, 84, 16, 350, 16]
  Data types: fb_after_stablility
<class 'int'>    5178
<class 'str'>     131
Name: count, dtype: int64
fe_after_stability: [225, 163, 348, 510, 348]
  Data types: fe_after_stability
<class 'int'>    5178
<class 'str'>     131
Name: count, dtype: int64

Sample neural data keys:
Type: <class 'dict'>
Keys (first 10): [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
Key types: [<class 'int'>, <class 'int'>, <class 'int'>, <class 'int'>, <class 'int'>]


In [15]:
# Helper functions for parallel processing
def is_trial_in_stability_range(fb_stability, fe_stability, trial_num):
    """Check if trial is within ANY stability range, handling multiple ranges properly"""
    # Wrap arguments in lists and create numpy arrays, then flatten
    fb_array = np.array([fb_stability]).flatten()
    fe_array = np.array([fe_stability]).flatten()
    
    # Get minimum length to ensure we don't go out of bounds
    min_len = min(len(fb_array), len(fe_array))
    
    # Check each stability range pair
    for i in range(min_len):
        fb_start = fb_array[i]
        fe_end = fe_array[i]
        
        # Skip invalid values
        if pd.isna(fb_start) or pd.isna(fe_end):
            continue
            
        try:
            # Check if trial falls within this range
            if int(fb_start) <= trial_num <= int(fe_end):
                return True  # Trial is within this stability range
        except (ValueError, TypeError):
            continue  # Skip this range if conversion fails
    
    return False  # Trial is not within any stability range

def parse_filename(filename):
    """Parse filename like 'fi210824a.0614' into components"""
    parts = filename.split('.')
    if len(parts) != 2:
        return None, None, None
    
    prefix = parts[0]  # 'fi210824a'
    trial_num = parts[1]  # '0614'
    
    if len(prefix) < 9:  # minimum: 'fi' + 6 digits + 'a'
        return None, None, None
    
    session = prefix[:-1]  # 'fi210824' (remove plexon session letter)
    plexon_session = prefix[-1]  # 'a'
    
    return session, plexon_session, int(trial_num)

def process_trial_chunk(args):
    """Process a chunk of trials - this function will run in parallel"""
    trial_chunk, cells_df_dict = args
    
    # Convert cells_df_dict back to DataFrame
    cells_df = pd.DataFrame(cells_df_dict)
    
    unified_data = []
    
    for _, trial_row in trial_chunk.iterrows():
        # Parse filename to get session info
        filename = trial_row['filename']
        session, plexon_session, trial_num = parse_filename(filename)
        
        if session is None:
            continue
        
        # Get neural data for this trial
        # neural_data = trial_row.get('neural_data', {})
        neural_data = trial_row['neural_data']
        if not isinstance(neural_data, dict):
            neural_data = {}
            raise ValueError(f"Unexpected neural_data format in trial '{filename}'")
        
        # Find cells that match session and plexon session
        session_cells = cells_df[
            (cells_df['session'] == session) #& 
            # (cells_df['plexon_session'] == plexon_session) !!!!!!!!! This could be the issue
        ]
        
        # Filter by stability range
        stable_cells = []
        for _, cell_row in session_cells.iterrows():
            if is_trial_in_stability_range(
                cell_row['fb_after_stablility'], 
                cell_row['fe_after_stability'], 
                trial_num
            ):
                stable_cells.append(cell_row)
        
        # Create one row per cell for this trial
        for cell_row in stable_cells:
            maestro_id_matlab = cell_row['maestro_ID']  # MATLAB 1-based ID
            
            # Fix MATLAB/Python indexing: MATLAB uses 1-based, Python uses 0-based
            # Neural data keys are 0-based (Python), maestro_ID in xlsx is 1-based (MATLAB)
            maestro_id_python = maestro_id_matlab - 1  # Convert to Python 0-based
            cell_neural_data = neural_data.get(maestro_id_python, [])
            
            # Create unified row
            unified_row = {
                # Cell information
                'cell_ID': cell_row['cell_ID'],
                'cell_type': cell_row['cell_type'], 
                'maestro_ID': maestro_id_matlab,  # Keep original MATLAB ID for reference
                'problem': cell_row['problem'],
                'grade': cell_row['grade'],
                
                # Core trial information
                'filename': trial_row['filename'],
                'trial_name': trial_row['trial_name'],
                'reaction_time': trial_row['reaction_time'],
                'go_cue': trial_row['go_cue'],
                'stop_cue': trial_row['stop_cue'], 
                'trial_failed': trial_row['trial_failed'],
                'ssd_len': trial_row['ssd_len'],
                'ssd_number': trial_row['ssd_number'],
                'type': trial_row['type'],
                
                # Additional trial information
                'first_relevant_saccade': trial_row['first_relevant_saccade'],
                'segs_durations': trial_row['segs_durations'],
                'segs_times': trial_row['segs_times'],
                'trial_length': trial_row['trial_length'],
                'screen_rotation': trial_row['screen_rotation'],
                'saccades': trial_row['saccades'],
                'blinks': trial_row['blinks'],
                'dir': trial_row['dir'],
                
                # Neural data for this specific cell
                'neural_data': cell_neural_data,
                
                # Additional useful columns
                'session': session,
                'plexon_session': plexon_session,
                'trial_number': trial_num,
                'trial_session': trial_row['trial_session']
            }
            
            unified_data.append(unified_row)
    
    return unified_data

# PARALLEL VERSION - Create the CORRECTED unified DataFrame with ProcessPoolExecutor
def create_unified_dataframe_parallel(trials_df, cells_df, chunk_size=100, max_workers=None):
    """
    Create a unified DataFrame with one row per cell-trial combination using parallel processing.
    
    Parameters:
    -----------
    trials_df : pd.DataFrame
        DataFrame with trial data
    cells_df : pd.DataFrame  
        DataFrame with cell information
    chunk_size : int
        Number of trials to process in each chunk
    max_workers : int
        Number of parallel workers (None = use CPU count)
    
    Returns:
    --------
    pd.DataFrame : Unified DataFrame with cell-trial combinations
    """
    print("Creating unified DataFrame with corrections (PARALLEL VERSION)...")
    print(f"Processing {len(trials_df)} trials and {len(cells_df)} cells...")
    
    # Convert cells_df to dict for pickling (required for ProcessPoolExecutor)
    cells_df_dict = cells_df.to_dict('records')
    
    # Split trials into chunks
    trial_chunks = []
    for i in range(0, len(trials_df), chunk_size):
        chunk = trials_df.iloc[i:i+chunk_size]
        trial_chunks.append((chunk, cells_df_dict))
    
    print(f"Split into {len(trial_chunks)} chunks of ~{chunk_size} trials each")
    
    # Determine number of workers
    if max_workers is None:
        max_workers = min(mp.cpu_count(), len(trial_chunks))
    
    print(f"Using {max_workers} parallel workers")
    
    # Process chunks in parallel
    all_unified_data = []
    all_unstable_trials = []    
    with ProcessPoolExecutor(max_workers=max_workers) as executor:
        # Submit all tasks
        futures = {executor.submit(process_trial_chunk, args): i 
                  for i, args in enumerate(trial_chunks)}
        
        # Collect results with progress bar
        for future in tqdm(as_completed(futures), total=len(futures), desc="Processing chunks"):
            chunk_results = future.result()
            all_unified_data.extend(chunk_results)
    
    # Create final DataFrame
    unified_df = pd.DataFrame(all_unified_data)

    print(f"\nUnified DataFrame created with {len(unified_df)} rows")
    print(f"Unique cells: {unified_df['cell_ID'].nunique()}")
    print(f"Unique trials: {unified_df['filename'].nunique()}")
    
    return unified_df

# Original sequential version (for comparison or fallback)
def create_unified_dataframe_corrected(trials_df, cells_df):
    """Sequential version - kept for fallback or comparison"""
    unified_data = []
    
    print("Creating unified DataFrame with corrections (SEQUENTIAL VERSION)...")
    print(f"Processing {len(trials_df)} trials and {len(cells_df)} cells...")
    
    for trial_idx, trial_row in tqdm(trials_df.iterrows(), total=len(trials_df), desc="Processing trials"):
        # Parse filename to get session info
        filename = trial_row['filename']
        session, plexon_session, trial_num = parse_filename(filename)
        
        if session is None:
            # raise ValueError(f"Could not parse filename: {filename}")
            continue
        
        # Get neural data for this trial
        # neural_data = trial_row.get('neural_data', {})
        neural_data = trial_row['neural_data']
        if not isinstance(neural_data, dict):
            # neural_data = {}
            raise ValueError(f"Unexpected neural_data format in trial '{filename}'")
        
        # Find cells that match session and plexon session
        session_cells = cells_df[
            (cells_df['session'] == session) #& 
            # (cells_df['plexon_session'] == plexon_session) !!!!!!!!!! This could be the issue
        ]
        
        # Filter by stability range
        stable_cells = []
        for _, cell_row in session_cells.iterrows():
            if is_trial_in_stability_range(
                cell_row['fb_after_stablility'], 
                cell_row['fe_after_stability'], 
                trial_num
            ):
                stable_cells.append(cell_row)
            else:
                error = f"Trial {trial_num} not in stability range for cell {cell_row['cell_ID']} ins session {session}"
                raise ValueError(error)
        
        # Create one row per cell for this trial
        for cell_row in stable_cells:
            maestro_id_matlab = cell_row['maestro_ID']  # MATLAB 1-based ID
            
            # Fix MATLAB/Python indexing: MATLAB uses 1-based, Python uses 0-based
            # Neural data keys are 0-based (Python), maestro_ID in xlsx is 1-based (MATLAB)
            maestro_id_python = maestro_id_matlab - 1  # Convert to Python 0-based
            cell_neural_data = neural_data.get(maestro_id_python, [])
            
            # Create unified row
            unified_row = {
                # Cell information
                'cell_ID': cell_row['cell_ID'],
                'cell_type': cell_row['cell_type'], 
                'maestro_ID': maestro_id_matlab,  # Keep original MATLAB ID for reference
                'problem': cell_row['problem'],
                
                # Core trial information
                'filename': trial_row['filename'],
                'trial_name': trial_row['trial_name'],
                'reaction_time': trial_row['reaction_time'],
                'go_cue': trial_row['go_cue'],
                'stop_cue': trial_row['stop_cue'], 
                'trial_failed': trial_row['trial_failed'],
                'ssd_len': trial_row['ssd_len'],
                'ssd_number': trial_row['ssd_number'],
                'type': trial_row['type'],
                
                # Additional trial information
                'first_relevant_saccade': trial_row['first_relevant_saccade'],
                'segs_durations': trial_row['segs_durations'],
                'segs_times': trial_row['segs_times'],
                'trial_length': trial_row['trial_length'],
                'screen_rotation': trial_row['screen_rotation'],
                'saccades': trial_row['saccades'],
                'blinks': trial_row['blinks'],
                'dir': trial_row['dir'],
                
                # Neural data for this specific cell
                'neural_data': cell_neural_data,
                
                # Additional useful columns
                'session': session,
                'plexon_session': plexon_session,
                'trial_number': trial_num,
                'trial_session': trial_row['trial_session']
            }
            
            unified_data.append(unified_row)
    
    unified_df = pd.DataFrame(unified_data)
    print(f"\nUnified DataFrame created with {len(unified_df)} rows")
    print(f"Unique cells: {unified_df['cell_ID'].nunique()}")
    print(f"Unique trials: {unified_df['filename'].nunique()}")
    
    return unified_df

# Test with sample data first - using parallel version
sample_df = df.sample(10, random_state=42)
print("Testing parallel version with 10 sample trials...")
unified_df_corrected = create_unified_dataframe_parallel(
    sample_df, 
    cell_db, 
    chunk_size=5, 
    max_workers=2
)
unified_df_corrected

Testing parallel version with 10 sample trials...
Creating unified DataFrame with corrections (PARALLEL VERSION)...
Processing 10 trials and 5309 cells...
Split into 2 chunks of ~5 trials each
Using 2 parallel workers


Processing chunks: 100%|██████████| 2/2 [00:00<00:00, 33.07it/s]



Unified DataFrame created with 284 rows
Unique cells: 284
Unique trials: 9


,cell_ID,cell_type,maestro_ID,problem,grade,filename,trial_name,reaction_time,go_cue,stop_cue,...,trial_length,screen_rotation,saccades,blinks,dir,neural_data,session,plexon_session,trial_number,trial_session
0,9214,msn,1,NaN,8,fi210726a.0613,CONT_R_SSD2,336.0,1033,1141.0,...,2184,0.0,"[[61, 136], [472, 534], [1369, 1443], [1803, 1...",None,0,[],fi210726,a,613,fi210726a
1,9215,msn,2,NaN,8,fi210726a.0613,CONT_R_SSD2,336.0,1033,1141.0,...,2184,0.0,"[[61, 136], [472, 534], [1369, 1443], [1803, 1...",None,0,"[600.8, 1033.85, 1472.38, 2177.5]",fi210726,a,613,fi210726a
2,9216,msn,3,NaN,8,fi210726a.0613,CONT_R_SSD2,336.0,1033,1141.0,...,2184,0.0,"[[61, 136], [472, 534], [1369, 1443], [1803, 1...",None,0,"[559.7, 1403.1, 1938.58]",fi210726,a,613,fi210726a
3,9217,msn,4,NaN,9,fi210726a.0613,CONT_R_SSD2,336.0,1033,1141.0,...,2184,0.0,"[[61, 136], [472, 534], [1369, 1443], [1803, 1...",None,0,[472.55],fi210726,a,613,fi210726a
4,9218,msn,5,NaN,9,fi210726a.0613,CONT_R_SSD2,336.0,1033,1141.0,...,2184,0.0,"[[61, 136], [472, 534], [1369, 1443], [1803, 1...",None,0,[187.68],fi210726,a,613,fi210726a
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
279,9887,msn,11,NaN,8,fi210824a.1625,GO_R,31.0,910,NaN,...,2061,0.0,"[[180, 249], [304, 432], [402, 466], [941, 996...","[218, 324]",0,"[416.15, 740.77, 854.5, 1122.65, 1124.35, 1226...",fi210824,a,1625,fi210824a
280,9888,msn,12,NaN,8,fi210824a.1625,GO_R,31.0,910,NaN,...,2061,0.0,"[[180, 249], [304, 432], [402, 466], [941, 996...","[218, 324]",0,"[1639.92, 1993.0, 2050.0, 2054.8]",fi210824,a,1625,fi210824a
281,9889,msn,13,NaN,8,fi210824a.1625,GO_R,31.0,910,NaN,...,2061,0.0,"[[180, 249], [304, 432], [402, 466], [941, 996...","[218, 324]",0,"[1015.9, 1816.35]",fi210824,a,1625,fi210824a
282,9890,msn,14,NaN,8,fi210824a.1625,GO_R,31.0,910,NaN,...,2061,0.0,"[[180, 249], [304, 432], [402, 466], [941, 996...","[218, 324]",0,"[417.62, 1545.32, 1763.12, 1771.48, 1871.22, 1...",fi210824,a,1625,fi210824a


In [16]:
unified_df_corrected['filename'].nunique(), sample_df['filename'].nunique()

(9, 10)

In [17]:
sample_df

,filename,trial_name,reaction_time,go_cue,stop_cue,trial_failed,first_relevant_saccade,segs_durations,segs_times,trial_length,...,ssd_number,screen_rotation,neural_data,saccades,blinks,dir,flags,type,trial_session,trial_number
81245,fi210726a.0613,CONT_R_SSD2,336.0,1033,1141.0,False,"[1369, 1443]","[500, 533, 108, 442, 600]","[0, 500, 1033, 1141, 1583, 2183]",2184,...,2.0,0.0,"{0: [], 1: [600.8, 1033.85, 1472.38, 2177.5], ...","[[61, 136], [472, 534], [1369, 1443], [1803, 1...",None,0,8206,CONT,fi210726a,0613
54788,fi210915a.1264,CONT_R_SSD1,312.0,997,1045.0,False,"[1309, 1381]","[500, 497, 48, 402, 700]","[0, 500, 997, 1045, 1447, 2147]",2148,...,1.0,0.0,"{0: [2.29, 17.38, 32.78, 89.21, 95.71, 111.01,...","[[242, 307], [1309, 1381], [1830, 1885]]",None,0,8206,CONT,fi210915a,1264
103766,fi211125a.1548,STOP_R_SSD4,331.0,1092,1320.0,True,"[1423, 1497]","[500, 592, 228, 700]","[0, 500, 1092, 1320, 2020]",2021,...,4.0,0.0,"{0: [101.05, 106.93, 132.75, 237.95, 300.2, 56...","[[70, 137], [384, 443], [1423, 1497], [1472, 1...",None,0,9222,STOP,fi211125a,1548
96336,fi211117a.1716,STOP_L_SSD4,171.0,1066,1294.0,True,"[1237, 1319]","[500, 566, 228, 700]","[0, 500, 1066, 1294, 1994]",1759,...,4.0,0.0,"{0: [112.21, 575.26, 800.58, 822.09, 969.81, 1...","[[47, 125], [1237, 1319], [1730, 1759]]",None,180,9222,STOP,fi211117a,1716
82875,fi210720a.0630,GO_R,408.0,1099,NaN,False,"[1507, 1580]","[500, 599, 550, 600]","[0, 500, 1099, 1649, 2249]",2250,...,NaN,0.0,"{0: [], 1: [], 2: [1951.77, 2131.17, 2156.68],...","[[103, 177], [1507, 1580]]",None,0,8206,GO,fi210720a,0630
45794,fi210922a.0385,GO_L,384.0,1060,NaN,False,"[1444, 1517]","[500, 560, 450, 700]","[0, 500, 1060, 1510, 2210]",2211,...,NaN,0.0,"{0: [47.97, 104.1, 127.25, 171.8, 185.63, 219....","[[74, 141], [187, 305], [272, 331], [440, 509]...","[109, 207]",180,8206,GO,fi210922a,0385
45735,fi210922a.1391,STOP_L_SSD2,376.0,929,1037.0,False,"[1305, 1360]","[500, 429, 108, 700]","[0, 500, 929, 1037, 1737]",1737,...,2.0,0.0,"{0: [40.8, 62.65, 103.35, 116.7, 128.85, 177.4...","[[221, 285], [805, 865], [1305, 1360]]",None,180,11278,STOP,fi210922a,1391
63331,fi211108a.1457,STOP_L_SSD4,161.0,975,1203.0,True,"[1136, 1214]","[500, 475, 228, 700]","[0, 500, 975, 1203, 1903]",1358,...,4.0,0.0,"{0: [220.48, 422.18, 443.25, 780.2, 839.6, 955...","[[71, 149], [460, 527], [924, 985], [1136, 121...",None,180,9222,STOP,fi211108a,1457
2211,fi210908a.0588,GO_R,173.0,1053,NaN,False,"[1226, 1299]","[500, 553, 450, 700]","[0, 500, 1053, 1503, 2203]",2204,...,NaN,0.0,"{0: [332.37, 631.78, 950.62, 987.1, 1502.77], ...","[[5, 78], [125, 239], [206, 263], [787, 848], ...","[46, 145]",0,8206,GO,fi210908a,0588
186,fi210824a.1625,GO_R,31.0,910,NaN,False,"[941, 996]","[500, 410, 450, 700]","[0, 500, 910, 1360, 2060]",2061,...,NaN,0.0,"{0: [492.67], 1: [619.62], 2: [1872.45], 3: []...","[[180, 249], [304, 432], [402, 466], [941, 996...","[218, 324]",0,8206,GO,fi210824a,1625


In [18]:
# Use the PARALLEL version for much faster processing
print("Using the PARALLEL unified DataFrame function...")
print("This function properly handles:")
print("1. MATLAB/Python indexing conversion")
print("2. Stability range filtering") 
print("3. Multiple stability ranges")
print("4. Parallel processing for speed")

# Run with full dataset - using optimal chunk size and max workers
print(f"\nProcessing full dataset with {len(df)} trials...")
unified_df = create_unified_dataframe_parallel(
    df, 
    cell_db, 
    chunk_size=200,  # Adjust based on memory vs speed tradeoff
    max_workers=None  # Use all available CPU cores
)

Using the PARALLEL unified DataFrame function...
This function properly handles:
1. MATLAB/Python indexing conversion
2. Stability range filtering
3. Multiple stability ranges
4. Parallel processing for speed

Processing full dataset with 110358 trials...
Creating unified DataFrame with corrections (PARALLEL VERSION)...
Processing 110358 trials and 5309 cells...
Split into 552 chunks of ~200 trials each
Using 20 parallel workers


Processing chunks: 100%|██████████| 552/552 [00:45<00:00, 12.03it/s]



Unified DataFrame created with 3123603 rows
Unique cells: 5121
Unique trials: 102383


In [19]:
# Analyze the unified DataFrame with additional columns
print("=== UNIFIED DATAFRAME ANALYSIS (WITH ADDITIONAL COLUMNS) ===")
print(f"Shape: {unified_df.shape}")
print(f"Memory usage: {unified_df.memory_usage(deep=True).sum() / 1e6:.1f} MB")

print(f"\nColumn info:")
print(f"Total columns: {len(unified_df.columns)}")
print(f"Columns: {list(unified_df.columns)}")

print(f"\nData summary:")
print(f"Unique cells: {unified_df['cell_ID'].nunique()}")
print(f"Unique trials: {unified_df['filename'].nunique()}")  
print(f"Unique sessions: {unified_df['session'].nunique()}")

print(f"\nCell types distribution:")
print(unified_df['cell_type'].value_counts())

print(f"\nTrial types in unified data:")
if 'type' in unified_df.columns:
    print(unified_df['type'].value_counts())

print(f"\nTrial name samples:")
trial_name_samples = unified_df['trial_name'].value_counts()
print(trial_name_samples.head(10))

print(f"\nNeural data statistics:")
neural_data_lengths = unified_df['neural_data'].apply(lambda x: len(x) if isinstance(x, (list, tuple)) else 0)
print(f"Mean spikes per trial: {neural_data_lengths.mean():.1f}")
print(f"Max spikes per trial: {neural_data_lengths.max()}")
print(f"Trials with no spikes: {(neural_data_lengths == 0).sum()}")

# Show sample rows with new columns
print(f"\nSample data with key columns:")
display_cols = ['cell_ID', 'cell_type', 'maestro_ID', 'filename', 'trial_name', 'type', 'reaction_time', 
                'first_relevant_saccade', 'trial_length', 'dir', 'problem']
available_cols = [col for col in display_cols if col in unified_df.columns]
print(f"Available columns: {available_cols}")
print(unified_df[available_cols].head(3))

# Check additional column data types and samples
print(f"\nAdditional column samples:")
additional_cols = ['segs_durations', 'segs_times', 'screen_rotation', 'saccades', 'blinks']
for col in additional_cols:
    if col in unified_df.columns:
        sample_val = unified_df[col].iloc[0]
        print(f"{col}: {type(sample_val).__name__} - {str(sample_val)[:100]}{'...' if len(str(sample_val)) > 100 else ''}")

=== UNIFIED DATAFRAME ANALYSIS (WITH ADDITIONAL COLUMNS) ===
Shape: (3123603, 27)
Memory usage: 4478.6 MB

Column info:
Total columns: 27
Columns: ['cell_ID', 'cell_type', 'maestro_ID', 'problem', 'grade', 'filename', 'trial_name', 'reaction_time', 'go_cue', 'stop_cue', 'trial_failed', 'ssd_len', 'ssd_number', 'type', 'first_relevant_saccade', 'segs_durations', 'segs_times', 'trial_length', 'screen_rotation', 'saccades', 'blinks', 'dir', 'neural_data', 'session', 'plexon_session', 'trial_number', 'trial_session']

Data summary:
Unique cells: 5121
Unique trials: 102383
Unique sessions: 84

Cell types distribution:
cell_type
pu msn     1587339
msn         940432
hfdp        230673
gpi          92350
lfd          91250
pu tan       63219
tan          51184
unknown      17668
lfdb         14507
fsn          10373
fiber         9590
bd            5050
fiber?        4991
fef           4308
tan            543
ctx            126
Name: count, dtype: int64

Trial types in unified data:
type
GO  

In [20]:
# Check for data quality and potential issues
print("=== DATA QUALITY CHECKS ===")

# Check for missing data
print("Missing values per column:")
missing_data = unified_df.isnull().sum()
print(missing_data[missing_data > 0])

# Check neural data distribution
print(f"\nNeural data statistics:")
neural_lengths = unified_df['neural_data'].apply(len)
print(f"Trials with neural data: {(neural_lengths > 0).sum()}/{len(unified_df)} ({(neural_lengths > 0).mean()*100:.1f}%)")

# Check problem cells
if 'problem' in unified_df.columns:
    problem_summary = unified_df['problem'].value_counts(dropna=False)
    print(f"\nProblem cell distribution:")
    print(problem_summary)

# Sample of actual neural data
print(f"\nSample neural data (first cell, first trial with data):")
sample_with_data = unified_df[neural_lengths > 0].iloc[0]
print(f"Cell ID: {sample_with_data['cell_ID']}")
print(f"Trial: {sample_with_data['trial_name']} ({sample_with_data['filename']})")
print(f"Neural data preview: {sample_with_data['neural_data'][:10]}...")  # First 10 spike times
print(f"Total spikes: {len(sample_with_data['neural_data'])}")

# Check trial distribution per cell
print(f"\nTrials per cell statistics:")
trials_per_cell = unified_df.groupby('cell_ID').size()
print(f"Mean trials per cell: {trials_per_cell.mean():.1f}")
print(f"Min trials per cell: {trials_per_cell.min()}")
print(f"Max trials per cell: {trials_per_cell.max()}")

# Show cells with most/least trials
print(f"\nCells with most trials:")
print(trials_per_cell.nlargest(5))
print(f"\nCells with fewest trials:")
print(trials_per_cell.nsmallest(5))

=== DATA QUALITY CHECKS ===
Missing values per column:
problem                   3119580
reaction_time              170374
stop_cue                  1727223
ssd_number                1727223
first_relevant_saccade     170374
saccades                     4156
blinks                    2544344
dtype: int64

Neural data statistics:
Trials with neural data: 2716619/3123603 (87.0%)

Problem cell distribution:
problem
NaN                                                 3119580
broken                                                 1361
check sorting with Mati                                1044
shape of waveform looks weird at last electrodes        593
synch problem                                           486
waveform looks different                                413
broken cell two peaks                                   126
Name: count, dtype: int64

Sample neural data (first cell, first trial with data):
Cell ID: 9867
Trial: CONT_L_SSD2 (fi210824a.0614)
Neural data preview: [ 154.42  

In [21]:
unified_df['trial_session'].value_counts()

trial_session
fi211104a    150097
fi211110a    149579
fi211017a    145260
fi211108a    141936
fi211025a    130217
              ...  
fi210718a       273
fi210628a       126
fi210704a       100
fi210705a        73
fi210629a         5
Name: count, Length: 84, dtype: int64

In [22]:
unified_df
print(unified_df[['cell_type', 'cell_ID']].drop_duplicates('cell_ID')['cell_type'].value_counts())

unified_df['grade'].value_counts()


cell_type
pu msn     2396
msn        1696
hfdp        441
lfd         153
gpi         138
tan          89
pu tan       83
lfdb         31
fsn          24
unknown      24
fiber        20
bd           10
fiber?        8
fef           6
ctx           1
tan           1
Name: count, dtype: int64


grade
8     1811870
9      672049
7      535445
6       81222
10      23017
Name: count, dtype: int64

In [34]:
unified_df[unified_df['cell_type'].isin(['msn']) & (unified_df['grade'] <= 8)]['cell_ID'].nunique()

1254

In [ ]:
# Save the unified DataFrame
save_path = Path.cwd().parent / 'data' / 'unified_cell_trial_data'
save_path.mkdir(exist_ok=True)

# Save as pickle for efficient loading
pickle_file = save_path / f'unified_{monkey}_cell_trial_data.pkl'
# unified_df.to_pickle(pickle_file)
print(f"Unified DataFrame saved to: {pickle_file}")

# # Also save a CSV version for easy inspection (but this will be larger)
# csv_file = save_path / f'unified_{monkey}_cell_trial_data.csv'
# # For CSV, convert neural_data to string representation to avoid issues
# csv_df = unified_df.copy()
# csv_df['neural_data'] = csv_df['neural_data'].apply(lambda x: str(x) if x else "[]")
# csv_df.to_csv(csv_file, index=False)
# print(f"CSV version saved to: {csv_file}")

# print(f"\nFile sizes:")
# print(f"Pickle: {pickle_file.stat().st_size / 1e6:.1f} MB") 
# print(f"CSV: {csv_file.stat().st_size / 1e6:.1f} MB")

# print(f"\nDataFrame ready for neural analysis!")
# print(f"Use: pd.read_pickle('{pickle_file}') to load the unified data")

Unified DataFrame saved to: /home/barak/Projects/population-analysis/data/unified_cell_trial_data/unified_fiona_cell_trial_data.pkl
